# Why the window sizes are what they are

This notebook records the diagnostic that changed the experimental design. It is exploration
only: every function it calls lives in `src/silentshift/`, and nothing is defined here that the
pipeline depends on.

The first pilot used 500-row windows against a 1000-row reference. Every detector was nearly
saturated on drift-free data — a Kolmogorov-Smirnov statistic of 1.0 where no change had been
injected. The question this notebook answers is why.

In [ ]:
import sys

sys.path.insert(0, "../src")

import matplotlib.pyplot as plt
import numpy as np

from silentshift.config import load_config
from silentshift.data.smd import load_machine
from silentshift.timeseries import autocorrelation_time, effective_sample_size

cfg = load_config("../configs/default.yaml")
machines = list(cfg.data.development_machines)[:6]
machines

## 1. How fast does the autocorrelation actually decay?

Mean absolute autocorrelation across the 38 signals, per lag.

In [ ]:
def decay_curve(x, lags):
    centred = x - x.mean(axis=0)
    var = np.mean(centred**2, axis=0)
    active = var > 1e-12
    centred, var = centred[:, active], var[active]
    return np.array(
        [np.mean(np.abs(np.mean(centred[k:] * centred[:-k], axis=0) / var)) for k in lags]
    )


lags = np.array([1, 5, 10, 25, 50, 100, 200, 300, 450, 600, 900, 1200])
fig, ax = plt.subplots(figsize=(9, 5))
for name in machines:
    series = load_machine(cfg.data.smd_root, name).train[:12000]
    ax.plot(lags, decay_curve(series, lags), marker="o", linewidth=1.3, label=name)

for level, style in ((0.3, "--"), (0.1, ":")):
    ax.axhline(level, color="black", linestyle=style, linewidth=1)
    ax.text(1250, level, f"  rho = {level}", va="center", fontsize=8)

ax.set_xlabel("lag (minutes)")
ax.set_ylabel("mean |autocorrelation| over 38 signals")
ax.set_title("SMD autocorrelation decays slowly and not monotonically")
ax.grid(alpha=0.25)
ax.legend(fontsize=8, frameon=False)
plt.show()

Two things to notice.

The decay is **slow**: correlation is still substantial at lag 100. And it is **not monotone** —
it dips around lag 300 and rises again near 600, which is a daily cycle rather than noise. That
non-monotonicity is why the estimator in `timeseries.py` takes the *first* crossing and is
documented as blunt: no single number summarises this curve honestly.

## 2. How many independent samples does a window actually contain?

This is the number that decided the design.

In [ ]:
rows = []
for name in machines:
    series = load_machine(cfg.data.smd_root, name).train[:12000]
    tau_03 = autocorrelation_time(series, threshold=0.3, max_lag=2000)
    tau_01 = autocorrelation_time(series, threshold=0.1, max_lag=2000)
    rows.append(
        {
            "machine": name,
            "tau@0.3": tau_03,
            "tau@0.1": tau_01,
            "eff. samples in a 500-row window": effective_sample_size(500, tau_03),
            "eff. samples in a 2500-row window": effective_sample_size(2500, tau_03),
        }
    )

import pandas as pd

pd.DataFrame(rows).set_index("machine")

A 500-row window holds a handful of effectively independent observations. No two-sample test
can distinguish distributions at that effective sample size, and the saturation seen in the
pilot is the expected consequence rather than a bug in any detector.

Thinning at rho = 0.1 would be more defensible statistically but demands steps of several
hundred rows and leaves almost no data. The project uses 0.3 and states the trade instead of
hiding it — see `docs/METHODOLOGY.md`.

## 3. What the saturation looked like

Scores of a KS detector on **drift-free** data, at the pilot geometry and at the final one.

In [ ]:
from silentshift.detectors import build

series = load_machine(cfg.data.smd_root, machines[0]).train

for label, ref_size, win_size, detector_name in [
    ("pilot   (ref 1000, win 500)", 1000, 500, "ks_max"),
    ("final   (ref 4000, win 2500)", 4000, 2500, "ks_max"),
    ("final + thinning", 4000, 2500, "ks_max_thinned"),
]:
    detector = build(detector_name, seed=0)
    detector.fit_reference(series[:ref_size])
    scores = [
        detector.score(series[start : start + win_size])
        for start in range(ref_size, ref_size + 8 * win_size, win_size)
    ]
    print(f"{label:30s} tau={detector.tau:4d}  scores={np.round(scores, 3)}")

These are all drift-free comparisons, so every score here is a false positive waiting to happen.
The pilot geometry pins the statistic near its maximum, which leaves no room above it for an
actual change to register.